# Deliverable 4: VLM-Gated DreamerV3 on Atari Boxing

**Authors:** Iqra Khurram (27100376) | Xeerak Azhar (27100310)

---

## What This Notebook Does

This notebook runs **ONLY the VLM-gated training** on Atari Boxing.

The baseline (no VLM) was already completed separately. This notebook:
1. Sets up the VLM gate with Gemini
2. Runs 500K steps of Boxing with VLM intervention
3. Generates comparison plots (using your baseline data from a previous run)

**Boxing is MUCH faster than Montezuma's Revenge:**
- Dense rewards (score on every successful punch)
- Episodes complete in seconds
- Meaningful scores within 50K-100K steps
- Total runtime: ~2-3 hours for 500K steps

You can safely interrupt training at any point and still get graphs.

In [1]:
# Cell 1 - Verify environment
import os, jax, jaxlib
import numpy as np

print('jax    :', jax.__version__)
print('jaxlib :', jaxlib.__version__)
print('numpy  :', np.__version__)
print('devices:', jax.devices())
assert len(jax.devices()) > 0, 'No GPU found!'
print('\n[OK] Environment verified.')

jax    : 0.4.33
jaxlib : 0.4.33
numpy  : 1.26.4
devices: [CudaDevice(id=0), CudaDevice(id=1)]

[OK] Environment verified.


In [2]:
# Cell 2 - Clone DreamerV3
import os, subprocess

REPO_DIR = '/kaggle/working/dreamerv3'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/danijar/dreamerv3.git', REPO_DIR], check=True)
    print('Cloned.')
else:
    print('Already exists.')
os.chdir(REPO_DIR)
print('CWD:', os.getcwd())

Already exists.
CWD: /kaggle/working/dreamerv3


In [3]:
# Cell 3 - Install requirements
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', 'requirements.txt'],
               check=True, capture_output=True)
print('[OK] DreamerV3 requirements installed.')

[OK] DreamerV3 requirements installed.


In [4]:
# Cell 4 - Verify imports
import elements
print('[OK] elements imported.')

[OK] elements imported.


In [5]:
# Cell 5 - Install Gemini SDK
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'google-generativeai', 'Pillow', '-q'],
               check=False, capture_output=True)
print('[OK] google-generativeai + Pillow ready.')

[OK] google-generativeai + Pillow ready.


In [ ]:
# Cell 6 - Load and TEST Gemini API key
GEMINI_API_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    print("[OK] Key from Kaggle Secrets.")
except Exception:
    GEMINI_API_KEY = "GEMINI_API_KEY"
    print("[WARN] Using fallback key.")

assert GEMINI_API_KEY, "No API key!"
print(f"Key present: {bool(GEMINI_API_KEY)}")

# CRITICAL: Find working model name
import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)

WORKING_MODEL = "gemini-1.5-flash"  # Default fallback
for model_name in ["gemini-2.0-flash-exp", "gemini-1.5-flash-latest", "gemini-1.5-flash-002", "gemini-1.5-flash"]:
    try:
        test_model = genai.GenerativeModel(model_name)
        resp = test_model.generate_content("Say OK")
        text = getattr(resp, 'text', '')
        if text:
            WORKING_MODEL = model_name
            print(f"[OK] {model_name} works: {text[:30]}")
            break
    except Exception as e:
        print(f"[FAIL] {model_name}: {str(e)[:60]}")

print(f"\n[SUCCESS] Will use: {WORKING_MODEL}")

[WARN] Using fallback key.
Key present: True


/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


[FAIL] gemini-2.0-flash-exp: 404 models/gemini-2.0-flash-exp is not found for API version
[FAIL] gemini-1.5-flash-latest: 404 models/gemini-1.5-flash-latest is not found for API vers
[FAIL] gemini-1.5-flash-002: 404 models/gemini-1.5-flash-002 is not found for API version
[FAIL] gemini-1.5-flash: 404 models/gemini-1.5-flash is not found for API version v1b

[SUCCESS] Will use: gemini-1.5-flash


In [7]:
# Cell 7 - Write vlm_gate.py with WORKING model name
import os

with open("/kaggle/working/vlm_gate.py", "w") as f:
    f.write("import json, time, numpy as np\n")
    f.write("from collections import deque\n\n")
    f.write("try:\n")
    f.write("    import google.generativeai as genai\n")
    f.write("    _GENAI = True\n")
    f.write("except Exception:\n")
    f.write("    _GENAI = False\n\n")
    f.write("ACTION_NAMES = {\n")
    f.write("    0:'NOOP',1:'FIRE',2:'UP',3:'RIGHT',4:'LEFT',5:'DOWN',\n")
    f.write("    6:'UPRIGHT',7:'UPLEFT',8:'DOWNRIGHT',9:'DOWNLEFT',10:'UPFIRE',\n")
    f.write("    11:'RIGHTFIRE',12:'LEFTFIRE',13:'DOWNFIRE',14:'UPRIGHTFIRE',\n")
    f.write("    15:'UPLEFTFIRE',16:'DOWNRIGHTFIRE',17:'DOWNLEFTFIRE',\n")
    f.write("}\n")
    f.write("NAME_TO_ACT = {v:k for k,v in ACTION_NAMES.items()}\n")
    f.write("VALID_ACTS = ['NOOP','FIRE','UP','RIGHT','LEFT','DOWN','UPRIGHT','UPLEFT','DOWNRIGHT','DOWNLEFT']\n\n")
    f.write("SIT_REWARD = {\n")
    f.write("    'punch':1.0,'hit':0.8,'jab':0.6,'dodge':0.3,'block':0.2,\n")
    f.write("    'opponent':0.4,'aggressive':0.5,'defensive':0.3,\n")
    f.write("    'getting hit':-0.6,'taking damage':-0.5,'losing':-0.4,\n")
    f.write("    'stuck':-0.3,'passive':-0.2,\n")
    f.write("}\n\n")
    f.write("def _parse_bonus(desc):\n")
    f.write("    dl = desc.lower()\n")
    f.write("    b = sum(v for k,v in SIT_REWARD.items() if k in dl)\n")
    f.write("    return float(np.clip(b, -1.0, 1.0))\n\n\n")
    f.write("class VLMGate:\n")
    f.write("    def __init__(self, api_key='', model_name='gemini-1.5-flash', threshold_z=2.5, queue_len=6,\n")
    f.write("                 cooldown_steps=80, reward_scale=0.25, warmup_steps=5000,\n")
    f.write("                 enabled=True, log_path='/kaggle/working/vlm_call_log.json'):\n")
    f.write("        self.model_name = model_name\n")
    f.write("        self.threshold_z = threshold_z\n")
    f.write("        self.queue_len = queue_len\n")
    f.write("        self.cooldown_steps = cooldown_steps\n")
    f.write("        self.reward_scale = reward_scale\n")
    f.write("        self.warmup_steps = warmup_steps\n")
    f.write("        self.enabled = enabled\n")
    f.write("        self.log_path = log_path\n")
    f.write("        self._queue = deque()\n")
    f.write("        self._rqueue = deque()\n")
    f.write("        self._prev = None\n")
    f.write("        self._sbuf = deque(maxlen=200)\n")
    f.write("        self._cd = 0\n")
    f.write("        self.total_vlm_calls = 0\n")
    f.write("        self.total_steps = 0\n")
    f.write("        self.total_triggers = 0\n")
    f.write("        self._log = []\n")
    f.write("        self._shist = []\n")
    f.write("        self._model = None\n")
    f.write("        self._fails = 0\n")
    f.write("        self._warmup_logged = False\n")
    f.write("        if enabled and api_key and _GENAI:\n")
    f.write("            try:\n")
    f.write("                genai.configure(api_key=api_key)\n")
    f.write("                self._model = genai.GenerativeModel(model_name)\n")
    f.write("                print(f'[VLMGate] READY | model={model_name} z={threshold_z} q={queue_len} cd={cooldown_steps} warm={warmup_steps}')\n")
    f.write("            except Exception as e:\n")
    f.write("                print(f'[VLMGate] Init failed: {e}')\n")
    f.write("        elif not api_key:\n")
    f.write("            print('[VLMGate] No API key.')\n\n")
    f.write("    def _surprise(self, frame):\n")
    f.write("        ff = frame.astype(np.float32)\n")
    f.write("        if self._prev is None:\n")
    f.write("            self._prev = ff\n")
    f.write("            return 0.0\n")
    f.write("        mse = float(np.mean((ff - self._prev)**2))\n")
    f.write("        self._prev = ff\n")
    f.write("        return mse\n\n")
    f.write("    def _should_trigger(self, s):\n")
    f.write("        self._sbuf.append(s)\n")
    f.write("        if len(self._sbuf) < 30: return False\n")
    f.write("        arr = np.array(self._sbuf, dtype=np.float32)\n")
    f.write("        std = float(arr.std())\n")
    f.write("        if std < 1e-6: return False\n")
    f.write("        z = (s - float(arr.mean())) / std\n")
    f.write("        return bool(z > self.threshold_z)\n\n")
    f.write("    def _call_vlm(self, frame, step, score, lives, hist):\n")
    f.write("        if self._model is None: return [], 0.0, 'no-model'\n")
    f.write("        if self._fails >= 10: return [], 0.0, 'too-many-fails'\n")
    f.write("        from PIL import Image\n")
    f.write("        import io\n")
    f.write("        try:\n")
    f.write("            fu8 = np.clip(frame, 0, 255).astype(np.uint8)\n")
    f.write("            if fu8.ndim == 2: pil = Image.fromarray(fu8, 'L').convert('RGB')\n")
    f.write("            elif fu8.shape[-1] == 1: pil = Image.fromarray(fu8[:,:,0], 'L').convert('RGB')\n")
    f.write("            else: pil = Image.fromarray(fu8)\n")
    f.write("            pil = pil.resize((160, 210), Image.NEAREST)\n")
    f.write("            hn = [ACTION_NAMES.get(int(a),'NOOP') for a in hist[-8:]]\n")
    f.write("            prompt = (\n")
    f.write("                'You are an expert Atari Boxing player. '\n")
    f.write("                'Analyze this boxing game screenshot and return ONLY valid JSON. '\n")
    f.write("                'No markdown, no backticks, no extra text.\\n'\n")
    f.write("                f'State: step={step} score={score:.0f} lives={lives}\\n'\n")
    f.write("                f'Recent actions: {hn}\\n'\n")
    f.write("                'Boxing strategy: punch when close, dodge when opponent attacks, move to position.\\n'\n")
    f.write("                'Return JSON with these fields:\\n'\n")
    f.write("                '  description: 1-2 sentences about what you see\\n'\n")
    f.write("                f'  actions: array of exactly {self.queue_len} action strings from {VALID_ACTS}\\n'\n")
    f.write("                '  reward_bonus: float from -1.0 to 1.0 (positive=landing punches, negative=getting hit)\\n'\n")
    f.write("            )\n")
    f.write("            buf = io.BytesIO()\n")
    f.write("            pil.save(buf, format='PNG')\n")
    f.write("            resp = self._model.generate_content([prompt, {'mime_type':'image/png','data':buf.getvalue()}])\n")
    f.write("            text = getattr(resp,'text','') or ''\n")
    f.write("            if not text: self._fails += 1; return [], 0.0, 'empty'\n")
    f.write("            self._fails = 0\n")
    f.write("            clean = text.strip()\n")
    f.write("            for p in ('```json','```'):\n")
    f.write("                if clean.startswith(p): clean = clean[len(p):]\n")
    f.write("            if clean.endswith('```'): clean = clean[:-3]\n")
    f.write("            clean = clean.strip()\n")
    f.write("            try: data = json.loads(clean)\n")
    f.write("            except json.JSONDecodeError:\n")
    f.write("                s,e = clean.find('{'), clean.rfind('}')\n")
    f.write("                if s==-1 or e==-1: return [], 0.0, 'parse-fail'\n")
    f.write("                data = json.loads(clean[s:e+1])\n")
    f.write("            acts = data.get('actions',[])\n")
    f.write("            if not isinstance(acts, list): acts = []\n")
    f.write("            va = []\n")
    f.write("            for nm in acts[:self.queue_len]:\n")
    f.write("                if isinstance(nm,str) and nm.strip().upper() in NAME_TO_ACT:\n")
    f.write("                    va.append(NAME_TO_ACT[nm.strip().upper()])\n")
    f.write("            while len(va) < self.queue_len: va.append(0)\n")
    f.write("            desc = str(data.get('description','')).strip()\n")
    f.write("            rb = data.get('reward_bonus', None)\n")
    f.write("            if rb is not None:\n")
    f.write("                try: bonus = float(np.clip(float(rb),-1,1))\n")
    f.write("                except: bonus = _parse_bonus(desc)\n")
    f.write("            else: bonus = _parse_bonus(desc)\n")
    f.write("            return va, bonus, desc\n")
    f.write("        except Exception as e:\n")
    f.write("            self._fails += 1\n")
    f.write("            print(f'[VLMGate] API err #{self._fails}: {type(e).__name__}: {e}')\n")
    f.write("            return [], 0.0, f'err:{type(e).__name__}'\n\n")
    f.write("    def get_action(self, frame, agent_action, step=0, score=0.0, lives=3, action_history=None):\n")
    f.write("        if not self.enabled: return agent_action, False, 0.0\n")
    f.write("        self.total_steps += 1\n")
    f.write("        action_history = action_history or []\n")
    f.write("        if step < self.warmup_steps:\n")
    f.write("            _ = self._surprise(frame)\n")
    f.write("            return agent_action, False, 0.0\n")
    f.write("        if not self._warmup_logged:\n")
    f.write("            print(f'[VLMGate] *** WARMUP DONE at step {step} -- gate ACTIVE ***')\n")
    f.write("            self._warmup_logged = True\n")
    f.write("        if self._queue:\n")
    f.write("            a = self._queue.popleft()\n")
    f.write("            b = self._rqueue.popleft() if self._rqueue else 0.0\n")
    f.write("            return a, True, b\n")
    f.write("        surprise = self._surprise(frame)\n")
    f.write("        if step % 500 == 0:\n")
    f.write("            self._shist.append({'step':step,'surprise':round(surprise,4)})\n")
    f.write("        if self._cd > 0:\n")
    f.write("            self._cd -= 1\n")
    f.write("            return agent_action, False, 0.0\n")
    f.write("        if self._should_trigger(surprise):\n")
    f.write("            self.total_triggers += 1\n")
    f.write("            va, rb, desc = self._call_vlm(frame, step, score, lives, action_history)\n")
    f.write("            if va:\n")
    f.write("                n = len(va)\n")
    f.write("                sc = rb * self.reward_scale\n")
    f.write("                rews = [sc*(1.0-i/n) for i in range(n)]\n")
    f.write("                self._queue.extend(va[1:])\n")
    f.write("                self._rqueue.extend(rews[1:])\n")
    f.write("                self._cd = self.cooldown_steps\n")
    f.write("                self.total_vlm_calls += 1\n")
    f.write("                self._log.append({'step':step,'surprise':round(surprise,2),\n")
    f.write("                    'reward_bonus':round(rb,3),'scaled_total':round(sum(rews),3),\n")
    f.write("                    'description':desc[:200],\n")
    f.write("                    'actions':[ACTION_NAMES.get(a,'NOOP') for a in va]})\n")
    f.write("                if self.total_vlm_calls % 5 == 0: self._save_silent()\n")
    f.write("                act_preview = [ACTION_NAMES.get(a,'?') for a in va[:3]]\n")
    f.write("                print(f'[VLMGate] step={step:>8} surp={surprise:7.2f} bonus={rb:+.2f} acts={act_preview}... calls={self.total_vlm_calls}')\n")
    f.write("                return va[0], True, rews[0]\n")
    f.write("        return agent_action, False, 0.0\n\n")
    f.write("    def reset_episode(self):\n")
    f.write("        self._queue.clear(); self._rqueue.clear(); self._prev = None; self._cd = 0\n\n")
    f.write("    def _save_silent(self):\n")
    f.write("        try:\n")
    f.write("            d = {'total_steps':self.total_steps,'total_vlm_calls':self.total_vlm_calls,\n")
    f.write("                 'total_triggers':self.total_triggers,\n")
    f.write("                 'call_pct':round(100*self.total_vlm_calls/max(1,self.total_steps),4),\n")
    f.write("                 'warmup_steps':self.warmup_steps,'threshold_z':self.threshold_z,\n")
    f.write("                 'model_name':self.model_name,\n")
    f.write("                 'surprise_history':self._shist[-500:],'calls':self._log}\n")
    f.write("            with open(self.log_path,'w') as f: json.dump(d,f,indent=2)\n")
    f.write("        except: pass\n\n")
    f.write("    def save_log(self):\n")
    f.write("        self._save_silent()\n")
    f.write("        print(f'[VLMGate] Log saved | {self.total_vlm_calls} calls / {self.total_steps} steps')\n")

print(f"[OK] vlm_gate.py written ({os.path.getsize('/kaggle/working/vlm_gate.py')} bytes)")

[OK] vlm_gate.py written (8881 bytes)


In [8]:
# Cell 8 - Write vlm_gym_wrapper.py
import os

wrapper_lines = []
wrapper_lines.append("import sys, numpy as np")
wrapper_lines.append("sys.path.insert(0, '/kaggle/working')")
wrapper_lines.append("from vlm_gate import VLMGate")
wrapper_lines.append("")
wrapper_lines.append("class VLMGymWrapper:")
wrapper_lines.append("    def __init__(self, env, api_key='', model_name='gemini-1.5-flash', threshold_z=2.5,")
wrapper_lines.append("                 queue_len=6, cooldown_steps=80, reward_scale=0.25, warmup_steps=5000):")
wrapper_lines.append("        self.env = env")
wrapper_lines.append("        self.gate = VLMGate(api_key=api_key, model_name=model_name, threshold_z=threshold_z,")
wrapper_lines.append("            queue_len=queue_len, cooldown_steps=cooldown_steps,")
wrapper_lines.append("            reward_scale=reward_scale, warmup_steps=warmup_steps,")
wrapper_lines.append("            enabled=True, log_path='/kaggle/working/vlm_call_log.json')")
wrapper_lines.append("        self._step = 0; self._score = 0.0; self._lives = 3")
wrapper_lines.append("        self._hist = []; self._obs = None")
wrapper_lines.append("        self.total_vlm_reward = 0.0; self.total_shaped = 0")
wrapper_lines.append("        for attr in ('observation_space','action_space','reward_range','metadata','spec','unwrapped'):")
wrapper_lines.append("            try: setattr(self, attr, getattr(env, attr))")
wrapper_lines.append("            except AttributeError: pass")
wrapper_lines.append("")
wrapper_lines.append("    def reset(self, **kw):")
wrapper_lines.append("        result = self.env.reset(**kw)")
wrapper_lines.append("        if isinstance(result, tuple) and len(result) == 2: obs, info = result")
wrapper_lines.append("        else: obs, info = result, {}")
wrapper_lines.append("        self._obs = np.array(obs)")
wrapper_lines.append("        self.gate.reset_episode()")
wrapper_lines.append("        self._step = 0; self._score = 0.0; self._lives = 3; self._hist = []")
wrapper_lines.append("        if isinstance(result, tuple) and len(result) == 2: return obs, info")
wrapper_lines.append("        return obs")
wrapper_lines.append("")
wrapper_lines.append("    def step(self, action):")
wrapper_lines.append("        frame = self._obs if self._obs is not None else np.zeros((210,160,3), np.uint8)")
wrapper_lines.append("        final, override, bonus = self.gate.get_action(")
wrapper_lines.append("            frame=frame, agent_action=int(action), step=self._step,")
wrapper_lines.append("            score=self._score, lives=self._lives, action_history=self._hist[-8:])")
wrapper_lines.append("        result = self.env.step(final)")
wrapper_lines.append("        if len(result) == 5:")
wrapper_lines.append("            obs, rew, term, trunc, info = result")
wrapper_lines.append("            done = term or trunc")
wrapper_lines.append("        elif len(result) == 4:")
wrapper_lines.append("            obs, rew, done, info = result")
wrapper_lines.append("        else: raise ValueError(f'Bad step return: {len(result)}')")
wrapper_lines.append("        shaped = float(rew) + bonus")
wrapper_lines.append("        if bonus != 0.0: self.total_vlm_reward += bonus; self.total_shaped += 1")
wrapper_lines.append("        self._obs = np.array(obs)")
wrapper_lines.append("        self._score += float(rew)")
wrapper_lines.append("        self._step += 1")
wrapper_lines.append("        self._hist.append(int(final))")
wrapper_lines.append("        if isinstance(info, dict):")
wrapper_lines.append("            if 'lives' in info: self._lives = int(info['lives'])")
wrapper_lines.append("            elif 'ale.lives' in info: self._lives = int(info['ale.lives'])")
wrapper_lines.append("        return obs, shaped, done, info")
wrapper_lines.append("")
wrapper_lines.append("    def close(self):")
wrapper_lines.append("        self.gate.save_log()")
wrapper_lines.append("        if self.total_shaped > 0:")
wrapper_lines.append("            print(f'[Wrapper] {self.total_shaped} shaped steps, total bonus={self.total_vlm_reward:+.2f}')")
wrapper_lines.append("        self.env.close()")
wrapper_lines.append("")
wrapper_lines.append("    def render(self, *a, **kw): return self.env.render(*a, **kw)")
wrapper_lines.append("    def seed(self, s=None):")
wrapper_lines.append("        if hasattr(self.env,'seed'): return self.env.seed(s)")
wrapper_lines.append("    def __getattr__(self, name): return getattr(self.env, name)")

with open("/kaggle/working/vlm_gym_wrapper.py", "w") as f:
    f.write("\n".join(wrapper_lines))

print(f"[OK] vlm_gym_wrapper.py written ({os.path.getsize('/kaggle/working/vlm_gym_wrapper.py')} bytes)")

[OK] vlm_gym_wrapper.py written (2892 bytes)


In [9]:
# Cell 9 - Patch DreamerV3 atari.py
import os, glob

candidates = glob.glob('/kaggle/working/dreamerv3/**/atari.py', recursive=True)
atari_path = None
for c in candidates:
    if 'embodied' in c and 'envs' in c:
        atari_path = c; break
if atari_path is None and candidates:
    atari_path = candidates[0]

assert atari_path, "atari.py not found!"
print(f"Found: {atari_path}")

with open(atari_path, 'r') as f:
    original = f.read()

marker = "VLM_GATE_D4_BOXING"
if marker in original:
    print("[OK] Already patched.")
else:
    patch = '''
# -- VLM_GATE_D4_BOXING --
import os as _os, sys as _sys
if _os.environ.get("VLM_GATE_ENABLED") == "1":
    _sys.path.insert(0, "/kaggle/working")
    try:
        from vlm_gym_wrapper import VLMGymWrapper as _W
        _patched = False
        for _mn in ("gymnasium", "gym"):
            try:
                _m = __import__(_mn)
            except: continue
            if hasattr(_m, "make"):
                _orig = _m.make
                def _wrapped(*a, __o=_orig, **kw):
                    return _W(__o(*a, **kw),
                        api_key=_os.environ.get("GEMINI_API_KEY",""),
                        model_name=_os.environ.get("GEMINI_MODEL_NAME","gemini-1.5-flash"),
                        threshold_z=float(_os.environ.get("VLM_THRESHOLD","2.5")),
                        queue_len=int(_os.environ.get("VLM_QUEUE_LEN","6")),
                        cooldown_steps=int(_os.environ.get("VLM_COOLDOWN","80")),
                        reward_scale=float(_os.environ.get("VLM_REWARD_SCALE","0.25")),
                        warmup_steps=int(_os.environ.get("VLM_WARMUP_STEPS","5000")))
                _m.make = _wrapped; _patched = True
        if _patched: print("[Atari] >>> VLM GATE ACTIVE <<<")
    except Exception as _e:
        import traceback; print(f"[Atari] VLM failed:"); traceback.print_exc()
# -- end VLM gate --

'''
    with open(atari_path, 'w') as f:
        f.write(patch + "\n" + original)
    print("[OK] atari.py patched.")

Found: /kaggle/working/dreamerv3/embodied/envs/atari.py
[OK] Already patched.


In [10]:
# Cell 10 - Quick VLM gate smoke test
import sys
sys.path.insert(0, "/kaggle/working")
import numpy as np
from vlm_gate import VLMGate

gate = VLMGate(
    api_key=GEMINI_API_KEY,
    model_name=WORKING_MODEL if 'WORKING_MODEL' in dir() else 'gemini-1.5-flash',
    threshold_z=2.5,
    queue_len=6,
    cooldown_steps=80,
    reward_scale=0.25,
    warmup_steps=20,
    enabled=True,
    log_path="/kaggle/working/vlm_test.json"
)

print("Running 80-step smoke test...")
for i in range(80):
    frame = np.random.randint(0, 255, (210, 160, 3), dtype=np.uint8)
    if i == 50: frame[:] = 255  # big surprise
    action, override, bonus = gate.get_action(frame, 0, step=i, score=0, lives=3, action_history=[0]*6)
    if override:
        print(f"  Step {i}: override! action={action} bonus={bonus:.3f}")

print(f"\nResult: {gate.total_steps} steps, {gate.total_triggers} triggers, {gate.total_vlm_calls} VLM calls")
if gate.total_vlm_calls > 0:
    print("[OK] VLM is working!")
else:
    print("[WARN] No VLM calls succeeded. Check Gemini model name.")

import os
if os.path.exists("/kaggle/working/vlm_test.json"):
    os.remove("/kaggle/working/vlm_test.json")

[VLMGate] READY | model=gemini-1.5-flash z=2.5 q=6 cd=80 warm=20
Running 80-step smoke test...
[VLMGate] *** WARMUP DONE at step 20 -- gate ACTIVE ***
[VLMGate] API err #1: NotFound: 404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
[VLMGate] API err #2: NotFound: 404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

Result: 80 steps, 2 triggers, 0 VLM calls
[WARN] No VLM calls succeeded. Check Gemini model name.


---
## VLM-Gated Training on Atari Boxing

**Game:** Boxing (dense rewards, fast episodes)  
**Steps:** 500K (you can interrupt at any point)  
**Expected:** Meaningful scores within 50K-100K steps

You can safely **interrupt** this cell and all analysis cells will still work.

In [ ]:
# Cell 12 - VLM-GATED DreamerV3 on BOXING (500K steps)
import os, sys, subprocess, shutil

LOGDIR_VLM = "/kaggle/working/dreamer_boxing_vlm"
os.makedirs(LOGDIR_VLM, exist_ok=True)

env = os.environ.copy()
env.pop("XLA_FLAGS", None)
env.pop("TF_XLA_FLAGS", None)
env["JAX_PLATFORMS"] = "cuda"
env["PYTHONNOUSERSITE"] = "1"

# VLM ON
env["VLM_GATE_ENABLED"] = "1"
env["GEMINI_API_KEY"] = GEMINI_API_KEY
env["GEMINI_MODEL_NAME"] = WORKING_MODEL
env["VLM_THRESHOLD"] = "2.5"
env["VLM_QUEUE_LEN"] = "6"
env["VLM_COOLDOWN"] = "80"
env["VLM_REWARD_SCALE"] = "0.25"
env["VLM_WARMUP_STEPS"] = "5000"

cmd = [
    sys.executable, "-u", "dreamerv3/main.py",
    "--logdir", LOGDIR_VLM,
    "--configs", "atari100k", "size12m",
    "--task", "atari_boxing",
    "--run.steps", "500000",
]

print("=" * 60)
print("VLM-GATED TRAINING ON BOXING")
print(f"Logdir: {LOGDIR_VLM}")
print(f"Model: {env['GEMINI_MODEL_NAME']}")
print("VLM log: /kaggle/working/vlm_call_log.json")
print()
print(">>> SAFE TO INTERRUPT — graphs still work <<<")
print("=" * 60)
print()

try:
    subprocess.run(cmd, env=env, check=True)
    print("\n[OK] VLM training complete!")
except KeyboardInterrupt:
    print("\n[INTERRUPTED] Training stopped by user.")
except subprocess.CalledProcessError as e:
    print(f"\n[ERROR] rc={e.returncode}")

if os.path.exists(LOGDIR_VLM):
    try:
        shutil.make_archive("/kaggle/working/backup_boxing_vlm", "zip", LOGDIR_VLM)
        print("Backup saved.")
    except: pass

VLM-GATED TRAINING ON BOXING
Logdir: /kaggle/working/dreamer_boxing_vlm
Model: gemini-1.5-flash
VLM log: /kaggle/working/vlm_call_log.json

>>> SAFE TO INTERRUPT — graphs still work <<<

---  ___                           __   ______ ---
--- |   \ _ _ ___ __ _ _ __  ___ _ \ \ / /__ / ---
--- | |) | '_/ -_) _` | '  \/ -_) '/\ V / |_ \ ---
--- |___/|_| \___\__,_|_|_|_\___|_|  \_/ |___/ ---
Replica: 0 / 1
Logdir: /kaggle/working/dreamer_boxing_vlm
Run script: train


/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.9.0+750d7f9)
[Powered by Stella]


[Atari] >>> VLM GATE ACTIVE <<<
Observations
  image            Space(uint8, shape=(96, 96, 1), low=0, high=255)
  reward           Space(float32, shape=(), low=-inf, high=inf)
  is_first         Space(bool, shape=(), low=False, high=True)
  is_last          Space(bool, shape=(), low=False, high=True)
  is_terminal      Space(bool, shape=(), low=False, high=True)
Actions
  action           Space(int32, shape=(), low=0, high=18)
Extras
  consec           Space(int32, shape=(), low=-2147483648, high=2147483647)
  stepid           Space(uint8, shape=(20,), low=0, high=255)
  dyn/deter        Space(float32, shape=(2048,), low=-inf, high=inf)
  dyn/stoch        Space(float32, shape=(32, 16), low=-inf, high=inf)
JAX devices (2): [cuda:0, cuda:1]
Policy devices: cuda:0
Train devices:  cuda:0
Initializing parameters...
Optimizer opt has 11,808,850 params:
     6,310,400 dyn
     2,256,001 dec
       853,503 val
       792,594 pol
       721,407 rew
       656,129 con
       218,816 enc
Done in

2026-05-07 15:31:07.690499: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778167868.168644     282 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778167868.330635     282 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778167869.443080     282 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778167869.443115     282 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778167869.443118     282 computation_placer.cc:177] computation placer alr


--------------------[Agent Step 6_200]--------------------
Metrics filtered by: 'score|length|fps|ratio|train/loss/|train/rand/'
train/loss/con 0.35 / train/loss/dyn 8.76 / train/loss/image 225.92 / train/loss/policy 1.7e-4 / train/loss/rep 8.76 / train/loss/repval 11.01 / train/loss/rew 5.5 / train/loss/value 2.36 / train/rand/action 1 / replay/replay_ratio 81.88 / fps/policy 12.35 / fps/train 946.57

Writing metrics: /kaggle/working/dreamer_boxing_vlm/metrics.jsonl
Writing metrics: /kaggle/working/dreamer_boxing_vlm/scores.jsonl
Stop JAX profiler

--------------------[Agent Step 8_360]--------------------
Metrics filtered by: 'score|length|fps|ratio|train/loss/|train/rand/'
episode/score -1 / episode/length 1785 / train/loss/con 0.04 / train/loss/dyn 7.7 / train/loss/image 64.15 / train/loss/policy 5.2e-3 / train/loss/rep 7.7 / train/loss/repval 10.23 / train/loss/rew 5.04 / train/loss/value 8.5 / train/rand/action 1 / replay/replay_ratio 260 / fps/policy 4.42 / fps/train 1130.77

W

---
## Analysis

**IMPORTANT:** For these graphs to work, you need your BASELINE data from a separate run.

Point the code below to your baseline logdir (should contain `metrics.jsonl`).

If you don't have a baseline yet, you'll see partial plots showing only the VLM run.

In [ ]:
# Cell 14 - Load metrics from both runs
import json, glob, os
import numpy as np

# CHANGE THIS to where your baseline is stored
LOGDIR_BASE = "/kaggle/working/dreamer_boxing_baseline"  # <-- UPDATE THIS PATH
LOGDIR_VLM = "/kaggle/working/dreamer_boxing_vlm"

def load_metrics(logdir):
    for fpath in [os.path.join(logdir, "metrics.jsonl")] + glob.glob(os.path.join(logdir, "**", "metrics.jsonl"), recursive=True):
        if os.path.exists(fpath):
            recs = []
            with open(fpath) as f:
                for line in f:
                    try: recs.append(json.loads(line.strip()))
                    except: pass
            if recs: return recs
    return []

def extract(recs, keys):
    steps, vals = [], []
    for r in recs:
        step = r.get("step", r.get("steps"))
        if step is None: continue
        for k in keys:
            if k in r and isinstance(r[k], (int, float)):
                steps.append(step); vals.append(float(r[k])); break
    return np.array(steps, dtype=np.float64), np.array(vals, dtype=np.float64)

SCORE_K = ["episode/score","episode_score","return","reward"]
DYN_K = ["train/loss/dyn","dyn_loss","loss/dyn"]
IMG_K = ["train/loss/image","image_loss","loss/image"]
LEN_K = ["episode/length","episode_length","length"]

base_r = load_metrics(LOGDIR_BASE)
vlm_r = load_metrics(LOGDIR_VLM)

base_ss, base_sv = extract(base_r, SCORE_K)
vlm_ss, vlm_sv = extract(vlm_r, SCORE_K)
base_ds, base_dv = extract(base_r, DYN_K)
vlm_ds, vlm_dv = extract(vlm_r, DYN_K)
base_is, base_iv = extract(base_r, IMG_K)
vlm_is, vlm_iv = extract(vlm_r, IMG_K)
base_ls, base_lv = extract(base_r, LEN_K)
vlm_ls, vlm_lv = extract(vlm_r, LEN_K)

print(f"{'Metric':<18} {'Baseline':>10} {'VLM':>10}")
print("-"*40)
for nm, b, v in [("Scores",base_sv,vlm_sv),("Dyn loss",base_dv,vlm_dv),
                  ("Img loss",base_iv,vlm_iv),("Ep length",base_lv,vlm_lv)]:
    print(f"{nm:<18} {len(b):>10} {len(v):>10}")

for label, sc in [("Baseline",base_sv),("VLM-gated",vlm_sv)]:
    if len(sc)>0: print(f"\n{label}: max={sc.max():.1f} mean={sc.mean():.1f}")
    else: print(f"\n{label}: no data")

In [ ]:
# Cell 15 - Load VLM call log
import json, os
import numpy as np

VLM_LOG = "/kaggle/working/vlm_call_log.json"
vlm_log = {"total_steps":0,"total_vlm_calls":0,"total_triggers":0,"call_pct":0,"calls":[],"surprise_history":[]}

if os.path.exists(VLM_LOG):
    with open(VLM_LOG) as f: vlm_log = json.load(f)
    calls = vlm_log.get("calls",[])
    print(f"[OK] VLM log loaded")
    print(f"  Model:    {vlm_log.get('model_name','unknown')}")
    print(f"  Steps:    {vlm_log.get('total_steps',0):,}")
    print(f"  Triggers: {vlm_log.get('total_triggers',0)}")
    print(f"  Calls:    {vlm_log.get('total_vlm_calls',0)}")
    print(f"  Rate:     {vlm_log.get('call_pct',0):.4f}%")
    if calls:
        bonuses = [c.get("reward_bonus",0) for c in calls]
        print(f"  Avg bonus: {np.mean(bonuses):+.3f}")
        print(f"  First: step {calls[0]['step']} | Last: step {calls[-1]['step']}")
        print(f"  Example: {calls[0].get('actions',[])} | {calls[0].get('description','')[:80]}")
else:
    print(f"[WARN] No VLM log at {VLM_LOG}")

In [ ]:
# Cell 16 - Plot: Score comparison
import matplotlib.pyplot as plt
import numpy as np

def smooth(a, w=20):
    if len(a)<w: return a, np.arange(len(a))
    s = np.convolve(a, np.ones(w)/w, mode="valid")
    return s, np.arange(w-1, len(a))

fig, ax = plt.subplots(figsize=(13,5))
if len(base_sv)>0:
    ax.plot(base_ss, base_sv, color="#B4B2A9", alpha=0.2, lw=0.5)
    s,i = smooth(base_sv); ax.plot(base_ss[i], s, color="#378ADD", lw=2.5, label=f"Baseline (max={base_sv.max():.0f})")
if len(vlm_sv)>0:
    ax.plot(vlm_ss, vlm_sv, color="#FFD580", alpha=0.2, lw=0.5)
    s,i = smooth(vlm_sv); ax.plot(vlm_ss[i], s, color="#F5A623", lw=2.5, label=f"VLM-gated (max={vlm_sv.max():.0f})")
if len(base_sv)==0 and len(vlm_sv)==0:
    ax.text(0.5,0.5,"No score data yet",transform=ax.transAxes,ha="center",va="center",fontsize=14)
ax.set_xlabel("Steps"); ax.set_ylabel("Score")
ax.set_title("Episode Score: Baseline vs VLM-Gated (Boxing)")
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("/kaggle/working/plot_scores_boxing.png", dpi=150); plt.show()
print("Saved: plot_scores_boxing.png")

In [ ]:
# Cell 17 - Combined: Losses + VLM analysis
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("DreamerV3 + VLM on Boxing: Full Analysis", fontsize=14)

calls = vlm_log.get("calls", [])
shist = vlm_log.get("surprise_history", [])

# Top-left: Dynamics loss
ax = axes[0,0]
if len(base_dv)>0:
    s,i = smooth(base_dv); ax.plot(base_ds[i], s, color="#378ADD", lw=2, label="Baseline")
if len(vlm_dv)>0:
    s,i = smooth(vlm_dv); ax.plot(vlm_ds[i], s, color="#F5A623", lw=2, label="VLM-gated")
ax.set_title("Dynamics Loss"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Top-right: Image loss
ax = axes[0,1]
if len(base_iv)>0:
    s,i = smooth(base_iv); ax.plot(base_is[i], s, color="#378ADD", lw=2, label="Baseline")
if len(vlm_iv)>0:
    s,i = smooth(vlm_iv); ax.plot(vlm_is[i], s, color="#F5A623", lw=2, label="VLM-gated")
ax.set_title("Image Loss"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Bottom-left: VLM trigger points
ax = axes[1,0]
if calls:
    ts = [c["step"] for c in calls]; tv = [c["surprise"] for c in calls]
    ax.scatter(ts, tv, color="#7F77DD", s=30, alpha=0.8)
    ax.set_title(f"VLM Trigger Points ({len(calls)} calls)")
else:
    ax.text(0.5,0.5,"No VLM calls",transform=ax.transAxes,ha="center",fontsize=12,color="gray")
    ax.set_title("VLM Trigger Points")
ax.set_xlabel("Step"); ax.set_ylabel("Surprise"); ax.grid(True, alpha=0.3)

# Bottom-right: VLM action distribution
ax = axes[1,1]
all_acts = []
for c in calls: all_acts.extend(c.get("actions",[]))
if all_acts:
    cnt = Counter(all_acts)
    labs, vals = zip(*sorted(cnt.items(), key=lambda x:-x[1]))
    ax.bar(labs, vals, color="#7F77DD", alpha=0.8, edgecolor="white")
    ax.set_title("VLM Action Distribution")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
else:
    ax.text(0.5,0.5,"No data",transform=ax.transAxes,ha="center",fontsize=12,color="gray")
    ax.set_title("VLM Action Distribution")
ax.set_xlabel("Action"); ax.set_ylabel("Count"); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout(); plt.savefig("/kaggle/working/plot_combined_boxing.png", dpi=150); plt.show()
print("Saved: plot_combined_boxing.png")

In [ ]:
# Cell 18 - Final summary
import os, glob
import numpy as np

print("=" * 65)
print("DELIVERABLE 4 — BOXING RESULTS")
print("=" * 65)
print()
print("Game    : Atari Boxing")
print("Model   : DreamerV3 (atari100k + size12m)")
print("VLM     : Gemini (model from Cell 6)")
print()

for label, sc, dy, im, ln in [
    ("Baseline", base_sv, base_dv, base_iv, base_lv),
    ("VLM-gated", vlm_sv, vlm_dv, vlm_iv, vlm_lv)]:
    print(f"  {label}:")
    if len(sc)>0: print(f"    Score: max={sc.max():.0f} mean={sc.mean():.1f} final={sc[-1]:.0f} eps={len(sc)}")
    else: print("    Score: no data")
    if len(dy)>0: print(f"    Dyn:   final={dy[-1]:.3f}")
    if len(im)>0: print(f"    Img:   final={im[-1]:.3f}")
    if len(ln)>0: print(f"    Len:   mean={ln.mean():.0f} max={ln.max():.0f}")
    print()

calls = vlm_log.get("calls",[])
print(f"VLM: {len(calls)} calls / {vlm_log.get('total_steps',0):,} steps / "
      f"{vlm_log.get('total_triggers',0)} triggers / {vlm_log.get('call_pct',0):.4f}% rate")
if calls:
    b = [c.get("reward_bonus",0) for c in calls]
    print(f"     Bonus: avg={np.mean(b):+.3f} range=[{min(b):+.3f}, {max(b):+.3f}]")

print()
print("Files:")
for p in sorted(glob.glob("/kaggle/working/plot*.png") + glob.glob("/kaggle/working/vlm_call_log.json") + glob.glob("/kaggle/working/backup_*.zip")):
    sz = os.path.getsize(p)
    print(f"  {os.path.basename(p):40s} {sz/1024:.0f} KB" if sz<1024*1024 else f"  {os.path.basename(p):40s} {sz/1024/1024:.1f} MB")

print()
print("=" * 65)
print("DONE — Ready for submission")
print("=" * 65)